# Similarity Analysis

Analisa a base mista e compartilha os mesmos dados finais entre Mahalanobis e PCA.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

from src.Similarity.Preprocessing import SimilarityPreprocessor
from src.Similarity.OneVsAll import SimilarityOneVsAll
from src.Similarity.SimilarityAnalyzer import SimilarityAnalyzer
from src.Plots.SimilarityDistanceHeatmapPlot import SimilarityDistanceHeatmapPlot
from src.Plots.SimilarityDendrogramPlot import SimilarityDendrogramPlot
from src.Plots.SimilaritySamples3DPlot import SimilaritySamples3DPlot
from src.Plots.SimilaritySamples2DPlot import SimilaritySamples2DPlot

datasetPath = Path("data/similarity/similarity_1s_180s.csv")
analysisOutputDirectory = Path("output/Similarity") / datasetPath.stem
intermediateDirectory = analysisOutputDirectory / "preprocessed"
plotDataPath = analysisOutputDirectory / "mahalanobis_plot_data.npz"


## 1. Preprocessing

Limpa e imputa os dados, com exportação intermediária opcional para a pasta desta análise.


In [ ]:
saveIntermediate = False
selectedFeatures = None
removedFeatures = ["FlowCount"]
nearConstantRatio = 0.999
normalization = "standard"

prepared = SimilarityPreprocessor(
    windowSeconds=1.0,
    nearConstantRatio=nearConstantRatio,
    removeZeroColumns=True,
    removeAllMissingColumns=True,
    removeConstantColumns=True,
    imputation="median",
    normalization=normalization,
).runFromFile(
    datasetPath=datasetPath,
    selectedFeatures=selectedFeatures,
    removedFeatures=removedFeatures,
    saveIntermediate=saveIntermediate, intermediateDirectory=intermediateDirectory,
)


## 2. One-vs-All

Seleciona a união do Top-K por classe usando todas as amostras preparadas.


In [ ]:
featuresPerAttack = 15
nEstimators = 300
oneVsAllRandomSeed = 42
nJobs = 1

selection = SimilarityOneVsAll(
    featuresPerAttack=featuresPerAttack,
    nEstimators=nEstimators,
    randomSeed=oneVsAllRandomSeed,
    nJobs=nJobs,
).run(prepared, saveIntermediate=saveIntermediate, intermediateDirectory=intermediateDirectory)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(selection["featureTable"])


## 3. Covariance and Centroids

Balanceia opcionalmente, padroniza e calcula os centroides e a covariância comum de Ledoit-Wolf.


In [ ]:
statisticsDownsample = "random"
statisticsRandomSeed = 42

analysis = SimilarityAnalyzer().run(
    selection=selection,
    outputDirectory=analysisOutputDirectory,
    downsample=statisticsDownsample,
    randomSeed=statisticsRandomSeed,
    saveIntermediate=saveIntermediate, intermediateDirectory=intermediateDirectory,
)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(analysis["centroids"])
    display(analysis["covariance"])


### 3.1 Mahalanobis Distance Matrix

Exibe as distâncias de Mahalanobis entre os centroides das classes.


In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(analysis["distanceMatrix"])


## 4. Heatmap and Dendrogram

Salva o mapa de calor e o dendrograma das distâncias na pasta da análise.


In [ ]:
heatmapPath = analysisOutputDirectory / "mahalanobis_distance_heatmap.png"
dendrogramPath = analysisOutputDirectory / "attack_distance_dendrogram.png"
plotDpi = 160
distanceMatrix = pd.read_csv(analysisOutputDirectory / "mahalanobis_distance_matrix.csv", index_col=0)

SimilarityDistanceHeatmapPlot().plot(distanceMatrix, heatmapPath, dpi=plotDpi)
SimilarityDendrogramPlot().plot(distanceMatrix, dendrogramPath, dpi=plotDpi)

display(Image(filename=str(heatmapPath)))
display(Image(filename=str(dendrogramPath)))


### 4.1 PCA Projection of Selected Attacks

Exibe todos os pontos finais translúcidos e seus centroides opacos no espaço de Mahalanobis projetado em três dimensões.


In [ ]:
plotOutputPath = analysisOutputDirectory / "attack_samples_pca_3d.png"
plotClasses = ["UDP", "SSDP", "SYN"]  # Edite os ataques; inclua "BENIGN" somente se desejar.
plotPointSize = 24
plotPointAlpha = 0.5
plotCentroidSize = 65
plotCentroidEdgeWidth = 0.8
plotElevation = 22
plotAzimuth = 40
plotDpi = 100
plotTitle = "PCA in Mahalanobis Space"

SimilaritySamples3DPlot().plotFromFile(
    dataPath=plotDataPath, outputPath=plotOutputPath, classes=plotClasses,
    pointSize=plotPointSize, pointAlpha=plotPointAlpha,
    centroidSize=plotCentroidSize, centroidEdgeWidth=plotCentroidEdgeWidth,
    elevation=plotElevation, azimuth=plotAzimuth,
    dpi=plotDpi, title=plotTitle,
)
display(Image(filename=str(plotOutputPath)))


### 4.2 PCA em duas componentes

Exibe PC1 e PC2 da mesma base salva, com ataques selecionáveis e centroides destacados.

In [ ]:
plotOutputPath = analysisOutputDirectory / "attack_samples_pca_2d.png"
plotClasses = ["UDP", "SSDP", "SYN"]  # Edite os ataques; inclua "BENIGN" somente se desejar.
plotPointSize = 24
plotPointAlpha = 0.5
plotCentroidSize = 65
plotCentroidEdgeWidth = 0.8
plotDpi = 100
plotTitle = "PCA in Mahalanobis Space"

SimilaritySamples2DPlot().plotFromFile(
    dataPath=plotDataPath, outputPath=plotOutputPath, classes=plotClasses,
    pointSize=plotPointSize, pointAlpha=plotPointAlpha,
    centroidSize=plotCentroidSize, centroidEdgeWidth=plotCentroidEdgeWidth,
    dpi=plotDpi, title=plotTitle,
)
display(Image(filename=str(plotOutputPath)))


In [ ]:
plotOutputPath = analysisOutputDirectory / "attack_samples_pca_2d.png"
plotClasses = ["UDP", "SSDP", "SYN", "BENIGN"]  # Edite os ataques; inclua "BENIGN" somente se desejar.
plotPointSize = 24
plotPointAlpha = 0.5
plotCentroidSize = 65
plotCentroidEdgeWidth = 0.8
plotDpi = 100
plotTitle = "PCA in Mahalanobis Space"

SimilaritySamples2DPlot().plotFromFile(
    dataPath=plotDataPath, outputPath=plotOutputPath, classes=plotClasses,
    pointSize=plotPointSize, pointAlpha=plotPointAlpha,
    centroidSize=plotCentroidSize, centroidEdgeWidth=plotCentroidEdgeWidth,
    dpi=plotDpi, title=plotTitle,
)
display(Image(filename=str(plotOutputPath)))
